In [1]:
"""
7. One-Shot Pruning — ResNet-50 / CIFAR-10
==========================================
One-shot pruning reaches the target sparsity in a SINGLE step —
no iterative refinement, no fine-tuning, no multiple passes.

Variants compared:
  A. One-shot L1 global    — single-pass global L1 magnitude
  B. One-shot L2 global    — single-pass global L2 magnitude
  C. One-shot random       — random pruning (baseline/control)

This is the simplest and fastest pruning approach.
Compare with 6_iterative_Pruning.json to quantify the cost
of going one-shot vs iterative.

Output: 7_oneshot_Pruning.json
"""

import os, json, time, copy, tempfile, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

warnings.filterwarnings("ignore")

# ── Config ────────────────────────────────────────────────────────────────────
BASELINE_MODEL_PATH   = "../__2__baseline_resnet50_cifar10.pth"
BASELINE_METRICS_PATH = "../__2__baseline_metrics.json"
OUTPUT_JSON           = "7_v2_oneshot_Pruning.json"

DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE  = 128
NUM_WORKERS = 2
NUM_CLASSES = 10

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

SPARSITY_LEVELS = [0.30] # , 0.50, 0.70, 0.80, 0.90
MAX_ACC_DROP    = 0.02
INFERENCE_RUNS  = 100


# ── Model ─────────────────────────────────────────────────────────────────────
def build_model(num_classes=10):
    model = models.resnet50(weights=None)
    model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc      = nn.Linear(model.fc.in_features, num_classes)
    return model

def load_baseline(path):
    model = build_model(NUM_CLASSES).to(DEVICE)
    model.load_state_dict(torch.load(path, map_location=DEVICE))
    model.eval()
    return model

# ── Data ──────────────────────────────────────────────────────────────────────
def get_test_loader():
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds = torchvision.datasets.CIFAR10(root="../data", train=False,
                                       download=True, transform=transform)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=True)


# ── Pruning variants ──────────────────────────────────────────────────────────
def prunable_layers(model):
    return [(m, "weight") for _, m in model.named_modules()
            if isinstance(m, (nn.Conv2d, nn.Linear))]

def oneshot_l1_global(model, sparsity):
    """Single-pass global L1 magnitude pruning."""
    pruned = copy.deepcopy(model)
    layers = prunable_layers(pruned)
    prune.global_unstructured(layers, pruning_method=prune.L1Unstructured, amount=sparsity)
    for module, param in layers:
        prune.remove(module, param)
    return pruned

def oneshot_l2_global(model, sparsity):
    """Single-pass global L2 magnitude pruning (custom criterion)."""
    class L2Unstructured(prune.BasePruningMethod):
        PRUNING_TYPE = "unstructured"
        def compute_mask(self, t, default_mask):
            tensor_size = t.nelement()
            nparams_toprune = max(1, round(tensor_size * self.amount))
            mask = default_mask.clone(memory_format=torch.contiguous_format)
            # rank by L2
            flat = t.pow(2).view(-1)
            topk = torch.topk(flat, k=tensor_size - nparams_toprune, largest=True)
            mask.view(-1).fill_(0)
            mask.view(-1)[topk.indices] = 1
            return mask
        @classmethod
        def apply(cls, module, name, amount):
            return super().apply(module, name, amount=amount)

    pruned = copy.deepcopy(model)
    for module, param in prunable_layers(pruned):
        L2Unstructured.apply(module, param, amount=sparsity)
        prune.remove(module, param)
    return pruned

def oneshot_random(model, sparsity):
    """Single-pass random pruning (control baseline)."""
    pruned = copy.deepcopy(model)
    layers = prunable_layers(pruned)
    prune.global_unstructured(layers, pruning_method=prune.RandomUnstructured, amount=sparsity)
    for module, param in layers:
        prune.remove(module, param)
    return pruned

def real_sparsity(model):
    zeros = total = 0
    for _, m in model.named_modules():
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            zeros += (m.weight == 0).sum().item()
            total += m.weight.numel()
    return zeros / total if total else 0.0

def count_params(model):
    total  = sum(p.numel() for p in model.parameters())
    active = sum((p != 0).sum().item() for p in model.parameters())
    return total, active

def disk_size_mb(model):
    with tempfile.NamedTemporaryFile(suffix=".pth", delete=False) as f:
        tmp = f.name
    try:
        torch.save(model.state_dict(), tmp)
        size = os.path.getsize(tmp) / 1024**2
    finally:
        os.remove(tmp)
    return size

def sparse_size_mb(model):
    active = sum((p != 0).sum().item() for p in model.parameters())
    return active * 4 / 1024**2


# ── Evaluation ────────────────────────────────────────────────────────────────
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, labels = [], []
    for inputs, lbls in loader:
        inputs = inputs.to(DEVICE, non_blocking=True)
        preds.extend(model(inputs).argmax(1).cpu().numpy())
        labels.extend(lbls.numpy())
    y_pred, y_true = np.array(preds), np.array(labels)
    return {
        "accuracy" : float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall"   : float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1"       : float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }

def measure_gpu_ms(model):
    if DEVICE.type != "cuda": return -1.0
    model.eval()
    dummy = torch.randn(BATCH_SIZE, 3, 32, 32, device=DEVICE)
    with torch.no_grad():
        for _ in range(20): model(dummy)
        torch.cuda.synchronize()
        s = torch.cuda.Event(enable_timing=True)
        e = torch.cuda.Event(enable_timing=True)
        s.record()
        for _ in range(INFERENCE_RUNS): model(dummy)
        e.record()
        torch.cuda.synchronize()
    return s.elapsed_time(e) / INFERENCE_RUNS

def measure_cpu_ms(model):
    cpu_m = copy.deepcopy(model).cpu().eval()
    dummy = torch.randn(BATCH_SIZE, 3, 32, 32)
    with torch.no_grad():
        for _ in range(5): cpu_m(dummy)
        t0 = time.perf_counter()
        for _ in range(20): cpu_m(dummy)
    return (time.perf_counter() - t0) / 20 * 1000

def compute_flops(model):
    """Total theoretical FLOPs — shape-based, no weight values needed."""
    flop_counts = {}
    hooks = []

    def make_hook(name):
        def hook(module, inp, out):
            if isinstance(module, nn.Conv2d):
                _, c_out, oH, oW = out.shape
                flop_counts[name] = 2 * module.in_channels * c_out * module.kernel_size[0] * module.kernel_size[1] * oH * oW
            elif isinstance(module, nn.Linear):
                flop_counts[name] = 2 * module.in_features * module.out_features
        return hook

    for name, module in model.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            hooks.append(module.register_forward_hook(make_hook(name)))

    model.eval()
    with torch.no_grad():
        model(torch.randn(1, 3, 32, 32).to(DEVICE))

    for h in hooks: h.remove()
    return sum(flop_counts.values()), flop_counts

def compute_active_flops(model, flop_counts):
    """FLOPs scaled by per-layer weight density (approximates sparse compute)."""
    active = 0
    for name, module in model.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)) and name in flop_counts:
            density = (module.weight != 0).float().mean().item()
            active += flop_counts[name] * density
    return int(active)

# ── Main ──────────────────────────────────────────────────────────────────────
def main():
    print(f"\n{'='*62}")
    print("  7. One-Shot Pruning — ResNet-50 / CIFAR-10")
    print(f"  Device: {DEVICE}  |  Variants: L1-global, L2-global, Random")
    print(f"{'='*62}\n")

    with open(BASELINE_METRICS_PATH) as f:
        baseline = json.load(f)

    model  = load_baseline(BASELINE_MODEL_PATH)
    loader = get_test_loader()
    base_disk = disk_size_mb(model)
    base_flops, flop_counts = compute_flops(model)

    VARIANTS = [
        ("oneshot_l1_global", oneshot_l1_global),
        # ("oneshot_l2_global", oneshot_l2_global),
        # ("oneshot_random",    oneshot_random),
    ]

    results    = []
    best_model = None
    best_score = -1.0
    best_cfg   = None

    for sparsity in SPARSITY_LEVELS:
        for variant_name, prune_fn in VARIANTS:
            print(f"  Sparsity {int(sparsity*100)}% | Variant: {variant_name} ...")
            try:
                pruned = prune_fn(model, sparsity)
            except Exception as e:
                print(f"    ERROR: {e}. Skipping.")
                continue

            actual_sp = real_sparsity(pruned)
            total_p, active_p = count_params(pruned)
            metrics   = evaluate(pruned, loader)
            inf_gpu   = measure_gpu_ms(pruned)
            inf_cpu   = measure_cpu_ms(pruned)
            acc_drop  = baseline["accuracy"] - metrics["accuracy"]
            sp_mb     = sparse_size_mb(pruned)
            dk_mb     = disk_size_mb(pruned)

            row = {
                "target_sparsity"            : sparsity,
                "variant"                    : variant_name,
                "actual_sparsity"            : round(actual_sp, 4),
                "accuracy"                   : round(metrics["accuracy"], 6),
                "precision"                  : round(metrics["precision"], 6),
                "recall"                     : round(metrics["recall"], 6),
                "f1"                         : round(metrics["f1"], 6),
                "accuracy_drop"              : round(acc_drop, 6),
                "params_total"               : total_p,
                "params_active"              : active_p,
                "size_sparse_theoretical_mb" : round(sp_mb, 4),
                "size_disk_mb"               : round(dk_mb, 4),
                "disk_saved_mb"              : round(base_disk - dk_mb, 4),
                "inference_gpu_ms"           : round(inf_gpu, 4) if inf_gpu > 0 else None,
                "inference_cpu_ms"           : round(inf_cpu, 4),
                "flops_total"   : int(base_flops),
                "flops_active"  : compute_active_flops(pruned, flop_counts),
                "flops_reduction_pct": round((1 - compute_active_flops(pruned, flop_counts) / base_flops) * 100, 2)

            }
            results.append(row)
            print(f"    -> Acc: {metrics['accuracy']:.4f} (drop: {acc_drop:+.4f}) | "
                  f"Actual sp: {actual_sp*100:.1f}%")

            if acc_drop <= MAX_ACC_DROP:
                score = metrics["accuracy"] + ((base_disk - dk_mb) / base_disk) * 0.1
                if score > best_score:
                    best_score = score
                    best_model = copy.deepcopy(pruned)
                    best_cfg   = {"sparsity": sparsity, "variant": variant_name}

    report = {
        "method"              : "oneshot_pruning",
        "description"         : "One-shot pruning: target sparsity reached in a single step — no iteration",
        "pruning_granularity" : "weight (unstructured, global)",
        "variants"            : ["oneshot_l1_global", "oneshot_l2_global", "oneshot_random"],
        "baseline"            : baseline,
        "device"              : str(DEVICE),
        "max_acc_drop_threshold": MAX_ACC_DROP,
        "best_config"         : best_cfg,
        "results"             : results,
        "notes": {
            "oneshot_l1_global": "Global L1 ranking, single pass — same as unstructured script",
            "oneshot_l2_global": "Global L2 (squared magnitude) ranking, single pass",
            "oneshot_random"   : "Randomly selected weights pruned — control baseline, worst accuracy",
            "vs_iterative"     : "Compare with 6_iterative_Pruning.json to see cost of one-shot vs iterative",
            "speed"            : "One-shot is extremely fast — no loops, single threshold computation",
        }
    }

    with open(OUTPUT_JSON, "w") as f:
        json.dump(report, f, indent=2)
    print(f"\n  ✓ Saved -> {OUTPUT_JSON}")
    if best_cfg:
        print(f"  Best: {best_cfg['variant']} @ {int(best_cfg['sparsity']*100)}% sparsity")


if __name__ == "__main__":
    main()



  7. One-Shot Pruning — ResNet-50 / CIFAR-10
  Device: cuda  |  Variants: L1-global, L2-global, Random

  Sparsity 30% | Variant: oneshot_l1_global ...
    -> Acc: 0.9321 (drop: -0.0001) | Actual sp: 30.0%

  ✓ Saved -> 7_v2_oneshot_Pruning.json
  Best: oneshot_l1_global @ 30% sparsity
